# Run the light feature-comparison modalities

A notebook fallback for `slurm/launch_feature_comp.sh`, covering the three cheapest modalities.
The counterpart to [`04b_run_training_manifests.ipynb`](04b_run_training_manifests.ipynb), which
does the same job for the full-cohort path.

| Modality | Where it runs |
|---|---|
| `stage`, `treatment`, `metburden` | **here** (1 CPU each) |
| `somatic`, `text`, `prs` | left to the SLURM arrays |

`run_feature_comp_task.py` forces `n_jobs=1` for any modality with fewer than 50 penalized
columns, which is why `launch_feature_comp.sh` sizes its `small` class at
`--cpus-per-task=1 --mem=4G`. Those modalities gain nothing from the cluster's parallelism —
they are single-core work queued behind the heavy jobs.

This notebook takes a **subset** of that `small` class. `somatic` is nominally in it, but its
design matrix is a wide gene-by-alteration panel whose width is data-dependent (`somatic_cols` is
derived at runtime from the `_AMP`/`_DEL`/`_SNV`/`_SV`/`_FUSION` suffixes), so it is left on
SLURM alongside `text` (dense embedding matrices) and `prs` (the only PCA path, k=1500).

### Running this alongside the SLURM arrays

Safe, and intended. `run_feature_comp_task.py` skips any scheme/event/modality whose four output
files already exist, so with `OVERWRITE = False` this notebook steps around whatever the arrays
have already finished, and vice versa.

The one caveat: that check happens once at task start. If the notebook and a SLURM task begin the
*same* triple within the same window, both run it and both write the same files. Outputs are
deterministic for a given input and hyperparameter grid, so that costs CPU rather than corrupting
anything.

Note the `small` SLURM array still covers `somatic`, so it should keep running — do not
`scancel` it the way you could if this notebook owned that whole class. To eliminate the overlap
instead, re-submit the array with a manifest pinned to `somatic` (the third TSV field) so the two
sides are disjoint.

### Why one subprocess per (event, modality), not `--modality all`

`run_feature_comp_task.py` has an in-process `--modality all` loop that loads the base frame once
and iterates, which is cheaper per modality than separate processes. It is deliberately **not**
used here: `all` means all *six*, which would pull in `text` and `prs` — exactly the work being
left to SLURM. The loader reads a modality's feature file only when that modality is requested, so
there is no way to get the shared-load benefit for a subset. One process per pair is also what the
`small` array itself does, so the per-process load cost matches current cluster behavior rather
than regressing against it, and one failing modality cannot take down the others.

Run this in a compute-backed Jupyter session — **not** on an ERISTwo login node.

In [ ]:
from __future__ import annotations

import os
import queue
import subprocess
import sys
import threading
import time
from pathlib import Path

from tqdm.auto import tqdm


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.py').is_file() and (candidate / 'pipelines').is_dir():
            return candidate
    raise RuntimeError(f'Could not find v2 root from {start}')


V2_ROOT = find_v2_root()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import schemes
from anchors import anchor_suffix
from pipelines.training.slurm_array_utils import _feature_file

print(f'Python:  {sys.executable}')
print(f'v2 root: {V2_ROOT}')

## Configuration

`MODALITIES` is a subset of `MODALITY_CLASS=small` in `slurm/array_feature_comp.sh`. `MAX_ITER` mirrors
the launcher's `COXNET_MAX_ITER` default and `BACKEND` its `COXNET_BACKEND` default — the grid
itself (alphas, l1 ratios, n_splits) is left at the script's defaults so results are comparable
with the array's. Change those in the script, not here.

In [ ]:
MODALITIES = ['stage', 'treatment', 'metburden']  # subset of MODALITY_CLASS=small
# 'somatic' is in the small class too, but is left to SLURM -- see the header.

ANCHOR = 'treatment'
N_JOBS = 1          # the script forces 1 for <50 penalized cols regardless
MAX_ITER = 1000     # COXNET_MAX_ITER default in launch_feature_comp.sh
BACKEND = 'threading'
OVERWRITE = False   # False leaves SLURM's finished work alone -- see the header

# Trial-run controls. Both None/empty for the full manifest.
MAX_TASKS = None            # e.g. 4 for a smoke run
SCHEMES_FILTER = None       # e.g. {'death_met'} (only 9 manifest rows)

MANIFEST = V2_ROOT / 'slurm' / 'slurm_manifests' / f'feature_comp_tasks{anchor_suffix(ANCHOR)}.tsv'

print(f'Anchor:     {ANCHOR}')
print(f'Modalities: {", ".join(MODALITIES)}')
print(f'Manifest:   {MANIFEST}')
print(f'Overwrite:  {OVERWRITE}')

## Preconditions

The manifest plus the feature files these three modalities read. A missing file here is the
difference between "the run failed" and "the run never had its inputs" — worth knowing before a
long cell starts.

`complete_somatic_data_df.csv.gz` and `complete_germline_data_df.csv.gz` are deliberately absent
from this list — those are the `somatic` and `prs` modalities, which stay on SLURM. This cell
does not raise.

In [ ]:
PRECONDITIONS = [
    ('manifest',           str(MANIFEST),                                                  'all'),
    ('cancer types',       _feature_file('cancer_type_df.csv.gz', ANCHOR),                  'all'),
    ('cancer stage',       _feature_file('cancer_stage_df.csv.gz', ANCHOR),                 'stage'),
    ('treatment by line',  _feature_file('categorical_treatment_data_by_line.csv.gz', ANCHOR), 'treatment'),
    ('met burden',         _feature_file('met_burden_df.csv.gz', ANCHOR),                   'metburden'),
]

missing = []
for label, path, used_by in PRECONDITIONS:
    ok = os.path.exists(path)
    if not ok:
        missing.append(label)
    print(f'[{"ok " if ok else "MISSING"}] {label:<18} (used by {used_by:<10}) {path}')

if missing:
    print(f'\n{len(missing)} input(s) missing: {", ".join(missing)}')
    print('Runs depending on them will fail. Inspect and decide.')
else:
    print('\nAll inputs present.')

## Build the task list

The manifest is `scheme<TAB>event`, with an optional third field pinning a row to one modality
(see `array_feature_comp.sh`). A pinned row is honoured here too, and dropped when it names a
modality left to SLURM (`somatic`, `text`, `prs`).

In [ ]:
SCHEME_ALIASES = {'icd3': 'icd3_post', 'icd4': 'icd4_post', 'phecode': 'phecode_post'}

tasks: list[tuple[str, str, str]] = []
rows = 0
skipped_pinned = 0

for line_number, raw in enumerate(MANIFEST.read_text().splitlines(), 1):
    if not raw.strip():
        continue
    fields = raw.split('\t')
    if len(fields) not in (2, 3) or not all(fields[:2]):
        raise ValueError(f'{MANIFEST}:{line_number}: expected scheme<TAB>event[<TAB>modality]')
    scheme = SCHEME_ALIASES.get(fields[0], fields[0])
    event = fields[1]
    pinned = fields[2] if len(fields) == 3 and fields[2] else None
    rows += 1

    if pinned is not None:
        if pinned in MODALITIES:
            tasks.append((scheme, event, pinned))
        else:
            skipped_pinned += 1
        continue

    if SCHEMES_FILTER and scheme not in SCHEMES_FILTER:
        continue
    tasks.extend((scheme, event, modality) for modality in MODALITIES)

if SCHEMES_FILTER:
    tasks = [task for task in tasks if task[0] in SCHEMES_FILTER]
if MAX_TASKS is not None:
    tasks = tasks[:MAX_TASKS]

print(f'Manifest rows: {rows}')
if skipped_pinned:
    print(f'Pinned to a modality left to SLURM: {skipped_pinned}')
print(f'Tasks: {len(tasks)}')
for task in tasks[:5]:
    print('  ', ':'.join(task))

## Pre-flight census

Reproduces the script's own skip test — the three grid files plus the held-out risk scores — so you
can see how much the SLURM array has already absorbed before starting a multi-hour cell.

Read-only: this uses `schemes.scheme_results_dir` rather than `get_output_dir`, which would create
the directories as a side effect.

In [ ]:
def output_paths(scheme: str, event: str, modality: str) -> list[str]:
    """The four files run_feature_comp_task checks before skipping a task."""
    results_dir = schemes.scheme_results_dir(scheme, ANCHOR)
    comp_dir = os.path.join(results_dir, 'feature_comps', event)
    return [
        os.path.join(comp_dir, f'{modality}_test.csv'),
        os.path.join(comp_dir, f'{modality}_val.csv'),
        os.path.join(comp_dir, f'{modality}_ipcw_reference.csv.gz'),
        os.path.join(schemes.feature_held_out_dir(scheme, event, ANCHOR), f'{modality}_risk_scores.csv'),
    ]


def is_done(scheme: str, event: str, modality: str) -> bool:
    return all(os.path.exists(path) for path in output_paths(scheme, event, modality))


def census(task_list: list[tuple[str, str, str]], header: str) -> list[tuple[str, str, str]]:
    done_by_modality: dict[str, int] = {modality: 0 for modality in MODALITIES}
    remaining: list[tuple[str, str, str]] = []
    for scheme, event, modality in task_list:
        if is_done(scheme, event, modality):
            done_by_modality[modality] = done_by_modality.get(modality, 0) + 1
        else:
            remaining.append((scheme, event, modality))

    total_done = len(task_list) - len(remaining)
    print(f'{header}: {total_done}/{len(task_list)} complete, {len(remaining)} remaining')
    for modality in MODALITIES:
        n_total = sum(1 for task in task_list if task[2] == modality)
        if n_total:
            print(f'  {modality:<10} {done_by_modality[modality]:>5}/{n_total} done')
    return remaining


remaining = census(tasks, 'Before run')

## Run

One `python -m pipelines.training.run_feature_comp_task` per task, with `cwd` set to `v2/`. Output
streams through a 15-second heartbeat — these go silent for long stretches, so silence would
otherwise be indistinguishable from a hang.

BLAS thread caps are pinned to 1 in the subprocess environment, matching `array_feature_comp.sh`.
Without them a Jupyter session on a many-core node oversubscribes badly.

A failure does not stop the queue; everything is reported at the end. Interrupting this cell is
safe — completed tasks are on disk and re-running skips them.

In [ ]:
def run_subprocess(cmd: list[str], label: str) -> dict:
    print(f"\n{'=' * 72}\n{label}\n$ {' '.join(cmd)}\n{'=' * 72}")
    started = time.perf_counter()
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    # Match array_feature_comp.sh: joblib owns the parallelism, BLAS stays single-threaded.
    for var in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
        env[var] = '1'
    process = subprocess.Popen(
        cmd, cwd=V2_ROOT, env=env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    output_queue: queue.Queue[str | None] = queue.Queue()

    def read_output() -> None:
        assert process.stdout is not None
        for line in process.stdout:
            output_queue.put(line)
        output_queue.put(None)

    threading.Thread(target=read_output, daemon=True).start()
    output_lines = []
    while True:
        try:
            line = output_queue.get(timeout=15)
        except queue.Empty:
            print(f'[still running] {label} ({time.perf_counter() - started:.0f}s)', flush=True)
            continue
        if line is None:
            break
        output_lines.append(line)
        print(line, end='', flush=True)
    returncode = process.wait()
    elapsed = time.perf_counter() - started
    print(f'[{label}] exit={returncode}, wall={elapsed:.1f}s')
    return {'returncode': returncode, 'wall_s': elapsed, 'stdout': ''.join(output_lines)}


def build_cmd(scheme: str, event: str, modality: str) -> list[str]:
    cmd = [sys.executable, '-m', 'pipelines.training.run_feature_comp_task',
           '--scheme', scheme, '--event', event, '--modality', modality,
           '--anchor', ANCHOR, '--n-jobs', str(N_JOBS),
           '--max-iter', str(MAX_ITER), '--backend', BACKEND]
    if OVERWRITE:
        cmd.append('--overwrite')
    return cmd


run_started = time.perf_counter()
results = []
for scheme, event, modality in tqdm(tasks, desc='feature comps'):
    label = f'{scheme}:{event}:{modality}'
    result = run_subprocess(build_cmd(scheme, event, modality), label)
    result.update(scheme=scheme, event=event, modality=modality)
    results.append(result)

run_elapsed = time.perf_counter() - run_started
print(f'\nRan {len(results)} task(s) in {run_elapsed / 60:.1f} min')

## Summary

Exit codes, then the census again so the end state on disk is visible.

A non-zero exit is not always a defect: `run_feature_comp_task` raises on events with too few
usable rows and logs the reason to `results/skipped_events/*.jsonl`. Check that file before
treating a failure as a bug.

In [ ]:
succeeded = [result for result in results if result['returncode'] == 0]
failed = [result for result in results if result['returncode'] != 0]

print(f'{len(succeeded)} succeeded, {len(failed)} failed')
if failed:
    print('\nFailures:')
    for result in failed:
        print(f"  {result['scheme']}:{result['event']}:{result['modality']} (exit {result['returncode']})")

slowest = sorted(results, key=lambda result: result['wall_s'], reverse=True)[:5]
if slowest:
    print('\nSlowest tasks:')
    for result in slowest:
        print(f"  {result['wall_s'] / 60:>6.1f} min  {result['scheme']}:{result['event']}:{result['modality']}")

print()
census(tasks, 'After run')